# Tableau exports

Turns the cleaned files into the `tab_*.csv` files that Tableau connects to. Each export has its own section.

| File | One row per | Owner |
|---|---|---|
| `tab_diet.csv` | country × year × food | A |
| `tab_health.csv` | country × year | A |
| `tab_diet_health_lag.csv` | country × diet year × lag × food × outcome | C |
| `tab_correlations.csv` | food × outcome × lag | C |

`tab_guidelines.csv` is made in `guidelines_compliance.ipynb`.

In [1]:
import pandas as pd
from pathlib import Path
from check_schema import check_schema

REF = Path('../data/reference')
CLEANED = Path('../data/cleaned')

countries = pd.read_csv(REF / 'countries.csv')

## tab_diet.csv

- Source: `data/cleaned/gdd_cleaned_long.csv`
- Output: one row per country × year × food
- Dropped: 2020 *(write why here)*

Step 1. Load the GDD long file.

In [3]:
# TODO: read gdd_cleaned_long.csv from CLEANED into a variable called gdd
gdd = pd.read_csv('../data/cleaned/gdd_cleaned_long.csv')

Step 2. Keep only the columns Tableau needs.

In [5]:
# TODO: keep iso3, year, factor_code, dietary_factor, unit,
#       intake_estimate, intake_lower_95, intake_upper_95
# Hint: gdd[[ ... ]]
cols = gdd[['iso3', 'year', 'factor_code', 'dietary_factor', 'unit', 'intake_estimate', 'intake_lower_95', 'intake_upper_95']]

Step 3. Rename the estimate columns to `intake`, `intake_low` and `intake_high`.

In [6]:
# TODO: rename the three estimate columns
# Hint: .rename(columns={'old': 'new', ...})
tableau_diet = cols.rename(columns={
    'intake_estimate': 'intake',
    'intake_lower_95': 'intake_low',
    'intake_upper_95': 'intake_high'
})

Step 4. Drop 2020, which only has 4 of the 47 foods.

In [17]:
# TODO: keep rows where year is not 2020
tableau_diet.head()
tableau_diet = tableau_diet[tableau_diet['year'] != 2020]

Step 5. Add country name and region from `countries.csv`.

In [18]:
# TODO: rename who_region to region in countries, then merge on iso3
# Hint: .merge(..., on='iso3', how='left', validate='many_to_one')

# This should print 0
# print(tableau_diet['region'].isna().sum())

countries = countries.rename(columns={
    'who_region': 'region'
})
tableau_diet = tableau_diet.merge(countries, on='iso3', how='left', validate='many_to_one')

print(tableau_diet['region'].isna().sum())

0


Step 6. Check and save. Expected: 60,865 rows × 10 columns, 185 countries, 1990–2018.

In [ ]:
# TODO: put the columns in this order:
#       iso3, country_name, region, year, factor_code, dietary_factor, unit, intake, intake_low, intake_high

tableau_diet = tableau_diet[['iso3', 'country_name', 'region', 'year', 'factor_code',
                            'dietary_factor', 'unit', 'intake', 'intake_low', 'intake_high']]

check_schema(tableau_diet, 'tableau_diet', grain=('iso3', 'year', 'factor_code'))
tableau_diet.to_csv(CLEANED / 'tab_diet.csv', index=False)




tab_diet: 60865 rows x 10 cols | iso3: 185 | years: 1990-2018
  OK


## tab_health.csv

- Source: `data/cleaned/who_gho_clean.csv`
- Output: one row per country × year
- Years: keep all (1980–2024). Diabetes starts in 1990 and premature deaths in 2000, so earlier years are empty for those columns. That's expected, so leave them as NaN.

Step 1. Load the WHO file and look at it.

In [26]:
# TODO: read who_gho_clean.csv from CLEANED into a variable called who
# Then look at it: who.head(), who.shape, who.isna().sum()
who = pd.read_csv('../data/cleaned/who_gho_clean.csv')
who.head()
# who.shape
# who.isna().sum()

,iso3,country_name,who_region,year,obesity_pct,obesity_pct_low,obesity_pct_high,diabetes_pct,diabetes_pct_low,diabetes_pct_high,premature_ncd_mort_pct,premature_ncd_mort_pct_low,premature_ncd_mort_pct_high
0,AFG,Afghanistan,Eastern Mediterranean,1980,0.97,0.44,1.91,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,Eastern Mediterranean,1981,1.04,0.49,1.99,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,Eastern Mediterranean,1982,1.12,0.54,2.07,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,Eastern Mediterranean,1983,1.20,0.61,2.17,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,Eastern Mediterranean,1984,1.29,0.68,2.27,NaN,NaN,NaN,NaN,NaN,NaN


Step 2. Check it's already one row per country × year.

The WHO cleaning notebook should have done this, but check before relying on it.

In [25]:
# TODO: count rows where the same iso3 + year appears more than once. It should be 0.
# Hint: who.duplicated(['iso3', 'year']).sum()
who.duplicated(['iso3', 'year']).sum()

np.int64(0)

Step 3. Take country name and region from `countries.csv`.

The WHO file has its own `country_name` and `who_region` columns. The team rule is that names and regions come only from `countries.csv`, so every file matches. Drop the WHO versions, then merge, the same way as in `tab_diet`.

`countries` already has `region` (you renamed it in the `tab_diet` section), so it doesn't need renaming again.

In [28]:
# TODO: drop country_name and who_region from who, and save the result as tableau_health
# Hint: .drop(columns=[...])
tableau_health = who.drop(columns=['country_name', 'who_region'])

# TODO: merge countries onto tableau_health on iso3 (left join, many_to_one)
tableau_health = tableau_health.merge(countries, on='iso3', how='left', validate='many_to_one')

# This should print 0
print(tableau_health['region'].isna().sum())


0


Step 4. Check and save. Expected: 8,955 rows × 13 columns, 199 countries, 1980–2024.

In [29]:
# TODO: put the columns in this order:
#       iso3, country_name, region, year,
#       obesity_pct, obesity_pct_low, obesity_pct_high,
#       diabetes_pct, diabetes_pct_low, diabetes_pct_high,
#       premature_ncd_mort_pct, premature_ncd_mort_pct_low, premature_ncd_mort_pct_high

tableau_health = tableau_health[['iso3', 'country_name', 'region', 'year', 'obesity_pct', 'obesity_pct_low', 'obesity_pct_high', 'diabetes_pct', 'diabetes_pct_low', 'diabetes_pct_high', 'premature_ncd_mort_pct', 'premature_ncd_mort_pct_low', 'premature_ncd_mort_pct_high' ]]

check_schema(tableau_health, 'tab_health')
tableau_health.to_csv(CLEANED / 'tab_health.csv', index=False)



tab_health: 8955 rows x 13 cols | iso3: 199 | years: 1980-2024
  OK


Later: if the group adds population or GDP, merge them in here on `iso3` + `year`.
They change every year, so they belong in this file, not in `countries.csv`.